## DataCo Smart Supply Chain Analytics

### An End-to-End Analytics Project Built on the SCOR Framework

#### Project Overview

This project builds a complete supply chain analytics pipeline using the DataCo Smart Supply Chain for Big Data Analysis dataset. The work follows the SCOR (Supply Chain Operations Reference) model, showing how advanced analytics can support strategic, tactical, and operational decision-making across the supply chain.

The project treats data science as an operational function within supply chain management. It focuses on translating structured supply chain data into outputs that serve three distinct audiences: planning teams, risk and reliability engineers, and senior decision-makers.

## Strategic Objectives (SCOR Alignment)

|Analytics Focus|	SCOR Phase(s)	|Business Value Demonstrated|
|---------------|-------------------|---------------------------|
|Demand Forecasting	|Plan / Deliver	|Improved resource and capacity planning.|
|Financial Impact Analysis	|Measure / Return	|Profitability, cost drivers, and ROI visibility.|
|Operational Risk and Resilience|	Source / Make / Deliver	|Reliability, disruption exposure, and mitigation.|
|Global Logistics Performance	|Deliver	|Bottleneck identification and logistics efficiency.|
|Customer-Centric Operations	|Plan / Deliver / Return	|Service levels, fulfillment quality, and market fit.|

- ###  Key Deliverables
    - Data Engineering and Modeling:
        - A single, cleaned analytical dataset built from the raw source tables.
        - Star-schema views exported as CSV/Parquet files for direct use in BI tools.
        - A complete data dictionary documenting source-to-target mappings, transformations, and column definitions.
    - Predictive Analytics:
        - Demand forecasting model with robust evaluation.
        - Scenario-driven planning implications for supply and capacity.
    - Risk and Reliability Modeling:
        - Predictive model estimating late-delivery probability.
        - Interpretable drivers of operational risk and service failure.
    - Decision-Maker KPI Framework
        - SCOR-aligned KPI definitions.
        - Power BI dashboards enabling strategic and operational monitoring.

## Dataset Context

The DataCo Smart Supply Chain dataset contains approximately 180,000 records and more than 50 features, spanning the period 2015-2019. It integrates transactional, logistics, customer, and financial data, enabling multi-level analysis across the supply chain.

Key characteristics include:

- Order-level logistics and fulfillment metrics.
- Financial indicators such as sales, discounts, and profits.
- Customer and geographic attributes supporting segmentation analysis.

## Strategic Rationale

This project demonstrates how an analytics team can structure work to serve both operational users and executives within a supply chain organization.

The architecture translates SCOR process categories into data models that feed directly into planning, risk assessment, and logistics optimization. The forecasting and risk models produce outputs that plug into Plan and Deliver workflows rather than existing as standalone exercises. The KPI framework and dashboards surface the metrics that matter for monthly and quarterly operational reviews.

The finished project reflects the structure, documentation, and decision-support logic expected from an embedded supply chain analytics function.

## Load DataCo Dataset

In [1]:
import numpy as np
import pandas as pd

# Load DataCo dataset
dataco_raw_path = "/kaggle/input/dataco-smart-supply-chain-for-big-data-analysis/DataCoSupplyChainDataset.csv"
dataco_raw = pd.read_csv(dataco_raw_path, encoding = "latin-1", low_memory = False)
dataco_raw.shape, dataco_raw.columns[:20]

((180519, 53),
 Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
        'Benefit per order', 'Sales per customer', 'Delivery Status',
        'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
        'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
        'Customer Lname', 'Customer Password', 'Customer Segment',
        'Customer State', 'Customer Street', 'Customer Zipcode'],
       dtype='object'))

## Data Dictionary Construction

As an initial data understanding and governance step, a programmatic data dictionary is generated directly from the raw DataCo dataset. Rather than relying on static documentation, this approach ensures that metadata accurately reflects the actual state of the data used in downstream analysis.

The dictionary captures, for each feature:

- **Data type**, supporting early validation and casting decisions.
- **Percentage of missing values**, highlighting data quality risks and informing imputation or exclusion strategies.
- **Cardinality** (number of unique values), useful for distinguishing identifiers, categorical dimensions, and continuous measures.
- **Representative example values**, enabling quick validation of each column.

Sorting the output by missingness and cardinality reveals high-risk and low-information features early, guiding feature selection and modeling priorities.

Overall, this step informs data cleaning, feature engineering, and BI modeling, while improving reproducibility and auditability.

In [2]:
def make_data_dictionary(df: pd.DataFrame) -> pd.DataFrame:
    output = []
    for c in df.columns:
        s = df[c]
        output.append({
            "column": c,
            "dtype": str(s.dtype),
            "missing_pct": float(s.isna().mean()),
            "n_unique": int(s.nunique(dropna = True)),
            "example": str(s.dropna().iloc[0]) if s.dropna().shape[0] else None
        })
    return pd.DataFrame(output).sort_values(["missing_pct","n_unique"], ascending = [False, True])

data_dictionary = make_data_dictionary(dataco_raw)
data_dictionary.head(25)

,column,dtype,missing_pct,n_unique,example
46,Product Description,float64,1.000000,0,None
43,Order Zipcode,float64,0.862397,609,99301.0
14,Customer Lname,object,0.000044,1109,Holloway
19,Customer Zipcode,float64,0.000017,995,725.0
11,Customer Email,object,0.000000,1,XXXXXXXXX
15,Customer Password,object,0.000000,1,XXXXXXXXX
50,Product Status,int64,0.000000,1,0
6,Late_delivery_risk,int64,0.000000,2,0
10,Customer Country,object,0.000000,2,Puerto Rico
16,Customer Segment,object,0.000000,3,Consumer


In [3]:
# Export data dictionary
data_dictionary_file = "data_dictionary.csv"
data_dictionary.to_csv(data_dictionary_file, index = False)
print("Saved:", data_dictionary_file)

Saved: data_dictionary.csv
